# GPR92 / LPAR5 mapping in pancreas — cleaned analysis notebook

This notebook is part of an integrated workflow (neonatal → chronic pancreatitis → spatial → integrated spatial).

**Local data (not committed):**
- `../data/raw/`
- `../data/processed/`

Outputs are written to `../outputs/`.

In [ ]:
# --- Environment & paths (portable) ---
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUT_DIR = Path("../outputs")
(OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "tables").mkdir(parents=True, exist_ok=True)

# Local project modules
import sys
sys.path.append(str(Path("..").resolve()))
from src import utils, preprocessing, scoring, plotting, spatial

In [ ]:
import scanpy as sc
import squidpy as sq
import spatialdata as sdata

print("Scanpy:", sc.__version__)
print("Squidpy:", sq.__version__)
print("Spatialdata:", sdata.__version__)


In [ ]:
import os

base = r"../data/raw/GSE198012_RAW"

for item in os.listdir(base):
    print(item)


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

# paths
base = r"../data/raw/GSE198012_RAW"

expr_path = base + r"\GSM5935792_ST_ND_rep1_filtered_feature_bc_matrix.h5"
coord_path = base + r"\GSM5935792_ND_rep1_tissue_positions_list.csv"

# read expression matrix (10x format)
adata = sc.read_10x_h5(expr_path)
adata.var_names_make_unique()

adata


In [ ]:
# read spatial coordinates
coords = pd.read_csv(coord_path, header=None)

coords.columns = [
    "barcode",
    "in_tissue",
    "array_row",
    "array_col",
    "pxl_row",
    "pxl_col"
]

coords.head()


In [ ]:
pd.read_csv(r"../data/raw/GSM5935792_ND_rep1_tissue_positions_list.csv", header=None)


In [ ]:
import pandas as pd

coord_path = r"../data/raw/GSM5935792_ND_rep1_tissue_positions_list.csv"
coords = pd.read_csv(coord_path, header=None)
coords.head()


In [ ]:
coord_path = r"../data/raw/GSM5935792_ND_rep1_tissue_positions_list.csv.gz"
coords = pd.read_csv(coord_path, header=None, compression="gzip")
coords.head()


In [ ]:
import scanpy as sc

expr_path = r"../data/raw/GSM5935792_ST_ND_rep1_filtered_feature_bc_matrix.h5"

adata = sc.read_10x_h5(expr_path)
adata.var_names_make_unique()
adata


In [ ]:
# assign column names
coords.columns = [
    "barcode",
    "in_tissue",
    "array_row",
    "array_col",
    "pxl_row",
    "pxl_col"
]

# set barcode as index
coords = coords.set_index("barcode")

# align coordinates to AnnData barcodes
coords = coords.loc[adata.obs_names]

# merge into obs
adata.obs = adata.obs.join(coords)

# add spatial coordinates
adata.obsm["spatial"] = adata.obs[["pxl_col", "pxl_row"]].values

adata


In [ ]:
# 3. QC metrics (creates total_counts, n_genes_by_counts)
sc.pp.calculate_qc_metrics(adata, inplace=True)

print(adata.obs.columns)


In [ ]:
print("Shape:", adata.shape)
print(adata.obs["in_tissue"].value_counts(dropna=False))


In [ ]:
# keep only spots on tissue
adata = adata[adata.obs["in_tissue"] == 1].copy()
print("After filter:", adata.shape)

# normalize + log
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# highly variable genes, but keep all genes
sc.pp.highly_variable_genes(
    adata,
    flavor="seurat_v3",
    n_top_genes=3000,
    subset=False
)

# work on HVGs for PCA/UMAP
adata_hvg = adata[:, adata.var["highly_variable"]].copy()

sc.pp.scale(adata_hvg, max_value=10)
sc.tl.pca(adata_hvg)
sc.pp.neighbors(adata_hvg)
sc.tl.umap(adata_hvg)

# copy UMAP back
adata.obsm["X_umap"] = adata_hvg.obsm["X_umap"]


In [ ]:
sc.pl.umap(adata, color="total_counts")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5,5))
plt.scatter(
    adata.obsm["spatial"][:, 0],
    adata.obsm["spatial"][:, 1],
    s=5
)
plt.gca().invert_yaxis()
plt.title("ND adipose – spot layout")
plt.show()


In [ ]:
import squidpy as sq

sq.gr.spatial_neighbors(adata, coord_type="grid")


In [ ]:
adata.uns["spatial_neighbors"].keys()


In [ ]:
genes = [
    # Macrophages
    "Adgre1", "Lyz2", "Csf1r",

    # Fibroblast / stromal
    "Col1a1", "Col3a1", "Dcn",

    # Inflammatory
    "Tnf", "Il1b", "Ccl2",

    # Vascular
    "Pecam1", "Kdr"
]

genes = [g for g in genes if g in adata.var_names]

sq.gr.spatial_autocorr(
    adata,
    mode="moran",
    genes=genes
)

adata.uns["moranI"]


In [ ]:
import scanpy as sc
import pandas as pd
import squidpy as sq
import numpy as np

base = r"../data/raw/GSE198012_RAW"

# HFD8 paths
expr_hfd8 = base + r"\GSM5935794_ST_HFD8_filtered_feature_bc_matrix.h5"
coord_hfd8 = base + r"\GSM5935794_HFD8_tissue_positions_list.csv.gz"

# load expression
adata_hfd8 = sc.read_10x_h5(expr_hfd8)
adata_hfd8.var_names_make_unique()

# load coordinates (gz)
coords = pd.read_csv(coord_hfd8, header=None, compression="gzip")

coords.columns = [
    "barcode",
    "in_tissue",
    "array_row",
    "array_col",
    "pxl_row",
    "pxl_col"
]

coords = coords.set_index("barcode")
coords = coords.loc[adata_hfd8.obs_names]

# attach coords
adata_hfd8.obs = adata_hfd8.obs.join(coords)
adata_hfd8.obsm["spatial"] = adata_hfd8.obs[["pxl_col", "pxl_row"]].values

# QC
sc.pp.calculate_qc_metrics(adata_hfd8, inplace=True)

# filter tissue spots
adata_hfd8 = adata_hfd8[adata_hfd8.obs["in_tissue"] == 1].copy()

adata_hfd8


In [ ]:
print("HFD8 shape:", adata_hfd8.shape)
print(adata_hfd8.obs["in_tissue"].value_counts(dropna=False))


In [ ]:
# normalize + log
sc.pp.normalize_total(adata_hfd8, target_sum=1e4)
sc.pp.log1p(adata_hfd8)

# HVGs (keep full gene set)
sc.pp.highly_variable_genes(
    adata_hfd8,
    flavor="seurat_v3",
    n_top_genes=3000,
    subset=False
)

adata_hvg = adata_hfd8[:, adata_hfd8.var["highly_variable"]].copy()

sc.pp.scale(adata_hvg, max_value=10)
sc.tl.pca(adata_hvg)
sc.pp.neighbors(adata_hvg)
sc.tl.umap(adata_hvg)

# copy embedding back
adata_hfd8.obsm["X_umap"] = adata_hvg.obsm["X_umap"]


In [ ]:
sc.pl.umap(adata_hfd8, color="total_counts")


In [ ]:
import squidpy as sq

sq.gr.spatial_neighbors(adata_hfd8, coord_type="grid")


In [ ]:
adata_hfd8.uns["spatial_neighbors"].keys()


In [ ]:
genes = [
    "Adgre1", "Lyz2", "Csf1r",
    "Col1a1", "Col3a1", "Dcn",
    "Il1b", "Tnf", "Ccl2",
    "Pecam1", "Kdr"
]

genes = [g for g in genes if g in adata_hfd8.var_names]

sq.gr.spatial_autocorr(
    adata_hfd8,
    mode="moran",
    genes=genes
)

adata_hfd8.uns["moranI"]


In [ ]:
import scanpy as sc
import pandas as pd
import squidpy as sq
import numpy as np

base = r"../data/raw/GSE198012_RAW"

# HFD8 paths
expr_hfd14 = base + r"\GSM5935795_ST_HFD14_filtered_feature_bc_matrix.h5"
coord_hfd14 = base + r"\GSM5935795_HFD14_tissue_positions_list.csv.gz"

# load expression
adata_hfd14 = sc.read_10x_h5(expr_hfd14)
adata_hfd14.var_names_make_unique()

# load coordinates (gz)
coords = pd.read_csv(coord_hfd14, header=None, compression="gzip")

coords.columns = [
    "barcode",
    "in_tissue",
    "array_row",
    "array_col",
    "pxl_row",
    "pxl_col"
]

coords = coords.set_index("barcode")
coords = coords.loc[adata_hfd14.obs_names]

# attach coords
adata_hfd14.obs = adata_hfd14.obs.join(coords)
adata_hfd14.obsm["spatial"] = adata_hfd14.obs[["pxl_col", "pxl_row"]].values

# QC
sc.pp.calculate_qc_metrics(adata_hfd14, inplace=True)

# filter tissue spots
adata_hfd14 = adata_hfd14[adata_hfd14.obs["in_tissue"] == 1].copy()

adata_hfd14

In [ ]:
# normalize + log
sc.pp.normalize_total(adata_hfd14, target_sum=1e4)
sc.pp.log1p(adata_hfd14)

# HVGs
sc.pp.highly_variable_genes(
    adata_hfd14,
    flavor="seurat_v3",
    n_top_genes=3000,
    subset=False
)

adata_hvg = adata_hfd14[:, adata_hfd14.var["highly_variable"]].copy()

sc.pp.scale(adata_hvg, max_value=10)
sc.tl.pca(adata_hvg)
sc.pp.neighbors(adata_hvg)
sc.tl.umap(adata_hvg)

adata_hfd14.obsm["X_umap"] = adata_hvg.obsm["X_umap"]


In [ ]:
sq.gr.spatial_neighbors(adata_hfd14, coord_type="grid")

genes = [
    "Adgre1", "Lyz2", "Csf1r",
    "Col1a1", "Col3a1", "Dcn",
    "Il1b", "Tnf", "Ccl2",
    "Pecam1", "Kdr"
]
genes = [g for g in genes if g in adata_hfd14.var_names]

sq.gr.spatial_autocorr(
    adata_hfd14,
    mode="moran",
    genes=genes
)

adata_hfd14.uns["moranI"]


In [ ]:
gpcr_genes = [
    # LPA / lipid sensing
    "Lpar1", "Lpar2", "Lpar3", "Lpar5",
    "Ffar2", "Ffar3",
    "Gpr132", "Gpr65",

    # Chemokine receptors
    "Ccr2", "Ccr5", "Cx3cr1",

    # Inflammatory GPCRs
    "C5ar1", "Ptger4", "P2ry6"
]


In [ ]:
def expressed_genes(adata, genes, min_frac=0.05):
    expr = {}
    for g in genes:
        if g in adata.var_names:
            x = adata[:, g].X
            if hasattr(x, "toarray"):
                x = x.toarray().flatten()
            frac = np.mean(x > 0)
            expr[g] = frac
    return pd.Series(expr).sort_values(ascending=False)

expr_nd = expressed_genes(adata, gpcr_genes)
expr_nd


In [ ]:
def expressed_genes(adata, genes, min_frac=0.05):
    expr = {}
    for g in genes:
        if g in adata.var_names:
            x = adata[:, g].X
            if hasattr(x, "toarray"):
                x = x.toarray().flatten()
            frac = np.mean(x > 0)
            expr[g] = frac
    return pd.Series(expr).sort_values(ascending=False)

expr_hfd8  = expressed_genes(adata_hfd8, gpcr_genes)
expr_hfd8


In [ ]:
def expressed_genes(adata, genes, min_frac=0.05):
    expr = {}
    for g in genes:
        if g in adata.var_names:
            x = adata[:, g].X
            if hasattr(x, "toarray"):
                x = x.toarray().flatten()
            frac = np.mean(x > 0)
            expr[g] = frac
    return pd.Series(expr).sort_values(ascending=False)

expr_hfd14 = expressed_genes(adata_hfd14, gpcr_genes)
expr_hfd14


In [ ]:
gpcr_keep = expr_hfd8[expr_hfd8 > 0.05].index.tolist()
gpcr_keep


In [ ]:
sq.gr.spatial_neighbors(adata_hfd8, coord_type="grid")

sq.gr.spatial_autocorr(
    adata_hfd8,
    mode="moran",
    genes=gpcr_keep
)

adata_hfd8.uns["moranI"]


In [ ]:
gpcr_keep = expr_hfd14[expr_hfd14 > 0.05].index.tolist()
gpcr_keep

In [ ]:
sq.gr.spatial_neighbors(adata_hfd14, coord_type="grid")

sq.gr.spatial_autocorr(
    adata_hfd14,
    mode="moran",
    genes=gpcr_keep
)

adata_hfd14.uns["moranI"]


In [ ]:
gpcr_keep = expr_nd[expr_nd > 0.05].index.tolist()
gpcr_keep

In [ ]:
sq.gr.spatial_neighbors(adata_nd, coord_type="grid")

sq.gr.spatial_autocorr(
    adata_nd,
    mode="moran",
    genes=gpcr_keep
)

adata_nd.uns["moranI"]

In [ ]:
def moran_gpcr(adata, condition):
    df = adata.uns["moranI"].copy()
    df["condition"] = condition
    df["gene"] = df.index
    return df[["gene", "condition", "pval_norm"]]

m_nd  = moran_gpcr(adata, "ND")
m_8   = moran_gpcr(adata_hfd8, "HFD8")
m_14  = moran_gpcr(adata_hfd14, "HFD14")

m_all = pd.concat([m_nd, m_8, m_14])
pivot = m_all.pivot(index="gene", columns="condition", values="pval_norm")
pivot


In [ ]:
gene = "C5ar1"  # example ONLY if it survives earlier steps

x, y = adata_hfd8.obsm["spatial"].T
expr = adata_hfd8[:, gene].X
if hasattr(expr, "toarray"):
    expr = expr.toarray().flatten()

plt.figure(figsize=(5,5))
plt.scatter(x, y, c=expr, s=5, cmap="magma")
plt.colorbar(label=gene)
plt.gca().invert_yaxis()
plt.title(f"HFD8 spatial: {gene}")
plt.show()



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_spatial_gene(adata, gene, title):
    x, y = adata.obsm["spatial"].T
    expr = adata[:, gene].X
    if hasattr(expr, "toarray"):
        expr = expr.toarray().flatten()

    plt.scatter(x, y, c=expr, s=5)
    plt.gca().invert_yaxis()
    plt.colorbar(label=gene)
    plt.title(title)
    plt.axis("off")

# ==== FIGURE 1A: C5ar1 spatial maps ====
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plot_spatial_gene(adata, "C5ar1", "ND – C5ar1")

plt.subplot(1, 3, 2)
plot_spatial_gene(adata_hfd8, "C5ar1", "HFD8 – C5ar1")

plt.subplot(1, 3, 3)
plot_spatial_gene(adata_hfd14, "C5ar1", "HFD14 – C5ar1")

plt.tight_layout()
plt.show()

# ==== FIGURE 1B: Csf1r spatial maps (macrophages) ====
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plot_spatial_gene(adata, "Csf1r", "ND – Csf1r (macrophages)")

plt.subplot(1, 3, 2)
plot_spatial_gene(adata_hfd8, "Csf1r", "HFD8 – Csf1r (macrophages)")

plt.subplot(1, 3, 3)
plot_spatial_gene(adata_hfd14, "Csf1r", "HFD14 – Csf1r (macrophages)")

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

def frac_positive(adata, gene, threshold="auto"):
    x = adata[:, gene].X
    if hasattr(x, "toarray"):
        x = x.toarray().flatten()
    if threshold == "auto":
        threshold = 0  # >0 counts = expressed; adjust later if needed
    return np.mean(x > threshold)

fractions = {
    "ND": frac_positive(adata, "C5ar1"),
    "HFD8": frac_positive(adata_hfd8, "C5ar1"),
    "HFD14": frac_positive(adata_hfd14, "C5ar1"),
}

fractions


In [ ]:
conds = list(fractions.keys())
vals = [fractions[c] for c in conds]

plt.figure(figsize=(4,4))
plt.bar(conds, vals)
plt.ylabel("Fraction of C5ar1⁺ spots")
plt.title("Expansion of C5ar1⁺ niches during obesity")
plt.show()


In [ ]:
def moran_for_gene(adata, gene):
    df = adata.uns["moranI"]
    if gene not in df.index:
        return np.nan
    return df.loc[gene, "I"]  # or "pval_norm" if you prefer that column

moran_c5ar1 = {
    "ND": moran_for_gene(adata, "C5ar1"),
    "HFD8": moran_for_gene(adata_hfd8, "C5ar1"),
    "HFD14": moran_for_gene(adata_hfd14, "C5ar1"),
}

moran_c5ar1


In [ ]:
conds = list(moran_c5ar1.keys())
vals = [moran_c5ar1[c] for c in conds]

plt.figure(figsize=(4,4))
plt.bar(conds, vals)
plt.ylabel("Moran's I (C5ar1)")
plt.title("C5ar1 spatial autocorrelation increases with obesity")
plt.show()


In [ ]:
def get_expr(adata, gene):
    x = adata[:, gene].X
    if hasattr(x, "toarray"):
        x = x.toarray().flatten()
    return x

# HFD14 data
x_hfd14, y_hfd14 = adata_hfd14.obsm["spatial"].T
c5 = get_expr(adata_hfd14, "C5ar1")
csf1r = get_expr(adata_hfd14, "Csf1r")

# define thresholds: above median or some percentile
c5_thr = np.percentile(c5[c5 > 0], 50) if np.any(c5 > 0) else 0
csf_thr = np.percentile(csf1r[csf1r > 0], 50) if np.any(csf1r > 0) else 0

niche_mask = (c5 > c5_thr) & (csf1r > csf_thr)

plt.figure(figsize=(5,5))
plt.scatter(x_hfd14, y_hfd14, s=5, alpha=0.2, label="Other spots")
plt.scatter(x_hfd14[niche_mask], y_hfd14[niche_mask], s=10, alpha=0.9, label="C5ar1⁺ Csf1r⁺ niche")
plt.gca().invert_yaxis()
plt.title("C5ar1⁺ macrophage niches (HFD14)")
plt.axis("off")
plt.legend()
plt.show()


In [ ]:
import numpy as np
import pandas as pd

def get_expr(adata, gene):
    x = adata[:, gene].X
    if hasattr(x, "toarray"):
        x = x.toarray().flatten()
    return x

def macrophage_mask(adata, q=60):
    csf1r = get_expr(adata, "Csf1r")
    lyz2  = get_expr(adata, "Lyz2")

    csf_thr = np.percentile(csf1r[csf1r > 0], q)
    lyz_thr = np.percentile(lyz2[lyz2 > 0], q)

    return (csf1r > csf_thr) & (lyz2 > lyz_thr)


In [ ]:
mask_nd    = macrophage_mask(adata)
mask_hfd8  = macrophage_mask(adata_hfd8)
mask_hfd14 = macrophage_mask(adata_hfd14)

print(mask_nd.mean(), mask_hfd8.mean(), mask_hfd14.mean())


In [ ]:
apoptosis_genes = [
    "Bax", "Bak1", "Casp3", "Casp8", "Casp9",
    "Fas", "Tnfrsf10b", "Bid", "Bcl2l11"
]


In [ ]:
necrosis_genes = [
    "Gsdmd", "Nlrp3", "Pycard", "Casp1",
    "Il1b", "Il18", "Ripk3", "Mlkl"
]


In [ ]:
proinf_genes = [
    "Tnf", "Il1b", "Il6", "Ccl2",
    "Ccl3", "Ccl4", "Nos2"
]


In [ ]:
antiinf_genes = [
    "Il10", "Tgfb1", "Arg1",
    "Mrc1", "Retnla", "Chil3"
]


In [ ]:
def score_program(adata, genes):
    genes = [g for g in genes if g in adata.var_names]
    if len(genes) == 0:
        return np.zeros(adata.n_obs)

    X = adata[:, genes].X
    if hasattr(X, "toarray"):
        X = X.toarray()
    return X.mean(axis=1)


In [ ]:
def score_all(adata):
    return {
        "apoptosis": score_program(adata, apoptosis_genes),
        "necrosis": score_program(adata, necrosis_genes),
        "proinf": score_program(adata, proinf_genes),
        "antiinf": score_program(adata, antiinf_genes),
    }


In [ ]:
scores_nd    = score_all(adata)
scores_hfd8  = score_all(adata_hfd8)
scores_hfd14 = score_all(adata_hfd14)


In [ ]:
def summarize(scores, mask):
    return {k: np.mean(v[mask]) for k, v in scores.items()}


In [ ]:
summary = pd.DataFrame([
    summarize(scores_nd, mask_nd),
    summarize(scores_hfd8, mask_hfd8),
    summarize(scores_hfd14, mask_hfd14),
], index=["ND", "HFD8", "HFD14"])

summary


In [ ]:
summary.plot(
    kind="bar",
    figsize=(7,4),
    ylabel="Mean program score",
    title="Macrophage-enriched spatial regions"
)


In [ ]:
for k, v in scores_nd.items():
    adata.obs[k + "_score"] = v


In [ ]:
for k, v in scores_hfd8.items():
    adata_hfd8.obs[k + "_score"] = v


In [ ]:
for k, v in scores_hfd14.items():
    adata_hfd14.obs[k + "_score"] = v


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_spatial_score(adata, score_key, title, vmax=None):
    x, y = adata.obsm["spatial"].T
    s = adata.obs[score_key].values

    plt.scatter(
        x, y,
        c=s,
        s=5,
        cmap="inferno",
        vmax=vmax
    )
    plt.gca().invert_yaxis()
    plt.axis("off")
    plt.colorbar(label=score_key)
    plt.title(title)


In [ ]:
vmax = np.percentile(
    np.concatenate([
        adata.obs["proinf_score"],
        adata_hfd8.obs["proinf_score"],
        adata_hfd14.obs["proinf_score"]
    ]),
    95
)

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plot_spatial_score(adata, "proinf_score", "ND – Pro-inflammatory", vmax)

plt.subplot(1,3,2)
plot_spatial_score(adata_hfd8, "proinf_score", "HFD8 – Pro-inflammatory", vmax)

plt.subplot(1,3,3)
plot_spatial_score(adata_hfd14, "proinf_score", "HFD14 – Pro-inflammatory", vmax)

plt.tight_layout()
plt.show()


In [ ]:
vmax = np.percentile(
    np.concatenate([
        adata.obs["necrosis_score"],
        adata_hfd8.obs["necrosis_score"],
        adata_hfd14.obs["necrosis_score"]
    ]),
    95
)

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plot_spatial_score(adata, "necrosis_score", "ND – Necrosis / pyroptosis", vmax)

plt.subplot(1,3,2)
plot_spatial_score(adata_hfd8, "necrosis_score", "HFD8 – Necrosis / pyroptosis", vmax)

plt.subplot(1,3,3)
plot_spatial_score(adata_hfd14, "necrosis_score", "HFD14 – Necrosis / pyroptosis", vmax)

plt.tight_layout()
plt.show()


In [ ]:
vmax = np.percentile(
    np.concatenate([
        adata.obs["antiinf_score"],
        adata_hfd8.obs["antiinf_score"],
        adata_hfd14.obs["antiinf_score"]
    ]),
    95
)

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plot_spatial_score(adata, "antiinf_score", "ND – Anti-inflammatory", vmax)

plt.subplot(1,3,2)
plot_spatial_score(adata_hfd8, "antiinf_score", "HFD8 – Anti-inflammatory", vmax)

plt.subplot(1,3,3)
plot_spatial_score(adata_hfd14, "antiinf_score", "HFD14 – Anti-inflammatory", vmax)

plt.tight_layout()
plt.show()


In [ ]:
mask = macrophage_mask(adata_hfd14)

plt.figure(figsize=(5,5))
x, y = adata_hfd14.obsm["spatial"].T

plt.scatter(x, y, s=4, alpha=0.15, label="All spots")
plt.scatter(x[mask], y[mask], s=6, alpha=0.8, label="Macrophage-rich")

plt.gca().invert_yaxis()
plt.axis("off")
plt.legend()
plt.title("Macrophage-enriched spatial regions (HFD14)")
plt.show()
